# **Processamento de Linguagem Natural [2025-Q3]**
Prof. Alexandre Donizeti Alves

### **PROJETO PRÁTICO** [LangChain + Grandes Modelos de Linguagem]


O **PROJETO PRÁTICO** deve ser feito utilizando o **Google Colab** com uma conta sua vinculada ao Gmail. O link do seu notebook armazenado no Google Drive e o link de um repositório no GitHub devem ser enviados usando o seguinte formulário:

> https://forms.gle/D4gLqP1iGgyn2hbH8


**IMPORTANTE**: A submissão deve ser feita até o dia **07/12 (domingo)** APENAS POR UM INTEGRANTE DA EQUIPE, até às 23h59. Por favor, lembre-se de dar permissão de ACESSO IRRESTRITO para o professor da disciplina.

### **EQUIPE**

---

**POR FAVOR, PREENCHER OS INTEGRANDES DA SUA EQUIPE:**


**Integrante 01:**

`Gabriel Victor Lima Gonçalves RA: 11202230490`

**Integrante 02:**

`Melissa Tami Vavassori RA: 11202231931`

**Integrante 03:**

`Vinicius Miranda da Silva RA: 11202231496`


**Repositorio com codigo completo no github:** https://github.com/Netreck/HireMatch-AI

**Site:** https://hirematch-ai-prod-820168498062.southamerica-east1.run.app



### **GRANDE MODELO DE LINGUAGEM (*Large Language Model - LLM*)**

---

Cada equipe deve selecionar um Grande Modelo de Linguagem (*Large Language Model - LMM*).



Por favor, informe os dados do LLM selecionada:

>


**LLM**:

> chatgpt-4.1 OpenAI

**Link para a documentação oficial**:

> https://platform.openai.com/docs/api-reference/introduction

### **API (Opcional)**
---

Por favor, informe os dados da API selecionada:

**API**: Remotive-API

**Site oficial**: https://remotive.com/

**Link para a documentação oficial**: https://github.com/remotive-com/remote-jobs-api






### **DESCRIÇÃO**
---

Implementar um `notebook` no `Google Colab` que faça uso do framework **`LangChain`** (obrigatório) e de um **LLM** aplicando, no mínimo, DUAS técnicas de PLN. As técnicas podem ser aplicada em qualquer córpus obtido a partir de uma **API** ou a partir de uma página Web.

O **LLM** e a **API** selecionados devem ser informados na seguinte planilha:

> https://docs.google.com/spreadsheets/d/1iIUZcwnywO7RuF6VEJ8Rx9NDT1cwteyvsnkhYr0NWtU/edit?usp=sharing

>
As seguintes técnicas de PLN podem ser usadas:

*   Correção Gramatical
*   Classificação de Textos
*   Análise de Sentimentos
*   Detecção de Emoções
*   Extração de Palavras-chave >    **UTILIZADO** para extrair tags html, tech stacks e soft skills da descrição da vaga
*   Tradução de Textos
*   Sumarização de Textos
*   Similaridade de Textos >    **UTILIZADO** como o principal componente para a nossa nota de similaridade com a vaga
*   Reconhecimento de Entidades Nomeadas
*   Sistemas de Perguntas e Respostas
>

**IMPORTANTE:** É obrigatório usar o e-mail da UFABC.


### **CRITÉRIOS DE AVALIAÇÃO**
---


Serão considerados como critérios de avaliação os seguintes pontos:

* Uso do framework **`LangChain`**.

* Escolha e uso de um **LLM**.

* Escolha e uso de uma **API** ou **Página Web**.

* Projeto disponível no Github.

* Apresentação (5 a 10 minutos).

* Criatividade no uso do framework **`LangChain`** em conjunto com o **LLM** e a **API**.




**IMPORTANTE**: todo o código do notebook deve ser executado. Código sem execução não será considerado.

### **IMPLEMENTAÇÃO**
---

# Estruturação > Definindo funções

## Pipeline Recebendo os dados da API e tratando 

## Pipeline Final Resumida

### Request da API filtrando areas tech

In [2]:
#Imports
import os
import re
import json
import requests
from pathlib import Path

Request API function

In [ ]:
import os
import json
import re
import requests

# Filtra areas tech
def pipeline_remotive_tech_jobs():
    TECH_CATEGORIES = [
        "software-development",
        "ai-ml",
        "data",
        "devops",
        "qa"
    ]

    def slugify_category(text: str) -> str:
        text = text.strip().lower()
        text = text.replace("/", " ")   
        text = re.sub(r"\s+", "-", text)   
        return text

    url = "https://remotive.com/api/remote-jobs"

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
    except Exception as e:
        print(f"Erro ao fazer request: {e}")
        return

    raw_jobs = response.json().get("jobs", [])

    tech_jobs = []
    for job in raw_jobs:
        category_raw = job.get("category", "")
        category_slug = slugify_category(category_raw)   # CONVERTE PRO SLUG CORRETO

        if category_slug in TECH_CATEGORIES:
            tech_jobs.append(job)

    save_path = "tech_jobs.json"
# Salva
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(tech_jobs, f, ensure_ascii=False, indent=4)

    print(f"✔ Arquivo salvo em: {os.path.abspath(save_path)}")
    print(f"✔ Salvo {len(tech_jobs)} tech jobs em tech_jobs.json")

### Recebe dados da API e faz tratamento inicial

Pipeline que trata os dados recebidos da api

In [61]:
import json
import re
from pathlib import Path

# >  RegexLimpar tags html - Extração
def clean_html(text: str) -> str:
    return re.sub(r"<[^>]+>", "", text or "")

# Efetivamente limpa a descrição recebida em html
def clean_jobs(raw_json: dict) -> list:
    raw_jobs = raw_json["jobs"]
    cleaned = []

    for job in raw_jobs:
        new_job = job.copy()
        new_job["description"] = clean_html(job.get("description", ""))
        cleaned.append(new_job)

    return cleaned

# carrega tech e soft_skills
def load_support_maps():
    folder = Path(".")

    tech_path = folder / "tech_stack.json"
    soft_path = folder / "soft_skills.json"

    with open(tech_path, "r", encoding="utf-8") as f:
        tech_map = json.load(f)

    with open(soft_path, "r", encoding="utf-8") as f:
        soft_map = json.load(f)

    tech_map = {k.lower(): [v.lower() for v in values] for k, values in tech_map.items()}
    soft_map = {k.lower(): [v.lower() for v in values] for k, values in soft_map.items()}

    return tech_map, soft_map

# Mapeia tech stack e soft skills retiradas do texto
def extract_skill_map(description: str, skill_map: dict):
    desc = description.lower()
    found = {}

    for macro, variations in skill_map.items():
        matches = []
        for var in variations:
            pattern = r"\b" + re.escape(var.lower()) + r"\b"
            if re.search(pattern, desc):
                matches.append(var)
                desc = re.sub(pattern, " ", desc)

        if matches:
            found[macro] = sorted(set(matches))

    desc = re.sub(r"\s+", " ", desc).strip()
    return desc, found

def process_job(job, tech_map, soft_map):
    desc = clean_html(job.get("description", "")).lower()

    desc, tech_matches = extract_skill_map(desc, tech_map)
    desc, soft_matches = extract_skill_map(desc, soft_map)

    job["description"] = desc
    job["tech_stacks_found"] = tech_matches
    job["soft_skills_found"] = soft_matches

    job.pop("tags", None)
    return job

# pipeline geral de limpeza
def pipeline(raw_path):
    raw_path = Path(raw_path)

    tech_map, soft_map = load_support_maps()

    with open(raw_path, "r", encoding="utf-8") as f:
        raw_jobs = json.load(f)

    processed = [process_job(job, tech_map, soft_map) for job in raw_jobs]

    output_path = Path("./jobs_processed.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=4, ensure_ascii=False)

    print(f"✔ Arquivo salvo em: {output_path.resolve()}")

### Embedding Pipe

In [43]:
import pandas
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
# Carrega dados da openAI e define modelo de embedding
load_dotenv() 
key = os.getenv("openai")
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key= key
    )

In [46]:
import ast
import json
import pandas as pd
from pathlib import Path
from langchain_openai import OpenAIEmbeddings


def generate_embeddings_df(path_json, model_name="text-embedding-3-large"):
    """
    Carrega o JSON processado, gera embeddings e salva como tech_jobs_com_embedding.csv na pasta atual.
    Retorna o dataframe resultante.
    """

    # Caminho de entrada
    path_json = Path(path_json)

    # Caminho de saída (sempre na pasta atual)
    output_csv = Path("./tech_jobs_com_embedding.csv")

    # Carrega JSON -> DataFrame
    df = pd.read_json(path_json)

    # Colunas que precisam converter str -> list
    cols_list = ["tech_stacks_found", "soft_skills_found"]

    for col in cols_list:
        df[col] = df[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    load_dotenv() 
    key = os.getenv("openai")
    embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key= key
    )
  

    # Gera embeddings
    df["embedding"] = df["description"].apply(embeddings.embed_query)

    # Salva saída
    df.to_csv(output_csv, index=False)

    print(f"✔ CSV salvo em: {output_csv.resolve()}")

    return df

### Nota currículo

Definindo curriculo

In [74]:
curriculo = """GABRIEL VICTOR LIMA GONC¸ ALVES
+55 (11)94924-4811 ⋄ Santo Andr´e, SP
gabrielvgonc@gmail.com ⋄ www.linkedin.com/in/gabriel-victor-71187b223
OBJECTIVE
Seeking career growth opportunities in the field of software development.
EDUCATION
Bachelor of Computer Science, Universidade Federal do ABC Expected Graduation: 2026
SKILLS
Main Technical Skills Python, SQL, ETL, Java, Git
Soft Skills Communication, Adaptability, Analytical Thinking, Proactivity
Github Portfolio https://github.com/Netreck
Languages English, Portuguese
PROFESSIONAL EXPERIENCE
Intern – Bank of America Jun 2025 – Present
VP Global Technology – Tech Rotation Program
• Developed test automation and tools to support QA teams in payment systems.
• Contributed to internal automation and process optimization projects.
• Technologies: Java, Python, Git, SQL, Node.js, HTML, CSS, Octane, QTest, Matera.
Data Intern – Vivo (Telefˆonica Brasil) Jan 2025 – Jun 2025
VP Engineering & Customer Services
• Generated, monitored, and analyzed operational KPIs focused on efficiency and customer experience.
• Built Machine Learning solutions to modernize and improve operational results.
• Technologies: Python, SQL, TensorFlow, Power BI, Excel.
EXTRACURRICULAR ACTIVITIES
Member – Green Team Hacker Club Jan 2024 – Jan 2025
Project Manager – Data Projects, Green Team Hacker Club Jan 2025 – Present
Federal University of ABC (UFABC) Santo Andr´e, SP
• Official student-led organization linked to UFABC.
• Participated in classes and projects related to Data Science.
• Defined and managed technology stack for projects, considering scalability, performance, and integration.
• Managed team activities including task delegation, progress tracking, and technical support.
• Worked with ETL, process automation (pipelines), SQL, PostgreSQL, Data Visualization (Seaborn,
Matplotlib), Data Modeling, Machine Learning models (Scikit-learn, TensorFlow), model deployment
via APIs, MLflow, and LLMs.
"""

Função que da nota ao curriculo

In [ ]:
import ast
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from langchain_openai import OpenAIEmbeddings
import re
import json

# Extrai skills do curriculo
def extract_skill_map(description: str, skill_map: dict):
    desc = description.lower()
    found = {}
    for macro, variations in skill_map.items():
        for v in variations:
            if v in desc:
                found.setdefault(macro, []).append(v)
    return desc, list(found.keys())


def load_support_maps(folder: Path):
    tech_path = folder / "tech_stack.json"
    soft_path = folder / "soft_skills.json"

    with tech_path.open("r", encoding="utf-8") as f:
        tech_map = json.load(f)

    with soft_path.open("r", encoding="utf-8") as f:
        soft_map = json.load(f)

    tech_map = {k.lower(): [v.lower() for v in vals] for k, vals in tech_map.items()}
    soft_map = {k.lower(): [v.lower() for v in vals] for k, vals in soft_map.items()}

    return tech_map, soft_map

# Cria logica de nota
def compute_final_score(similarity_raw, sim_min, sim_max,
                        job_techs, job_soft,
                        curr_techs, curr_soft):

    similarity_norm = (similarity_raw - sim_min) / (sim_max - sim_min + 1e-9)
    similarity_norm = max(0, min(1, similarity_norm))

    total_tech = len(job_techs)
    tech_val = (
        sum(1 for macro in job_techs if macro in curr_techs) / total_tech
        if total_tech > 0 else 1
    )

    total_soft = len(job_soft)
    soft_val = (
        sum(1 for macro in job_soft if macro in curr_soft) / total_soft
        if total_soft > 0 else 1
    )

    final = (
        0.7 * similarity_norm +
        0.1 * tech_val +
        0.2 * soft_val
    ) * 100

    return round(final, 2)

# Adiciona cada nota ao curriculo
def match_curriculo_com_df(path_df, curriculo, model_name="text-embedding-3-large"):
    path_df = Path(path_df)
    folder = path_df.parent

    tech_map, soft_map = load_support_maps(folder)

    if path_df.suffix == ".csv":
        df = pd.read_csv(path_df)
    else:
        df = pd.read_json(path_df)

    for col in ["tech_stacks_found", "soft_skills_found", "embedding"]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )

    load_dotenv() 
    key = os.getenv("openai")
    embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key= key
    )
    curr_emb = embeddings.embed_query(curriculo)

    curr_clean = re.sub(r"\s+", " ", curriculo).strip().lower()
    _, curr_techs = extract_skill_map(curr_clean, tech_map)
    _, curr_soft = extract_skill_map(curr_clean, soft_map)

    df["embedding_similarity"] = df["embedding"].apply(
        lambda v: float(cosine_similarity(
            np.array(curr_emb).reshape(1, -1),
            np.array(v).reshape(1, -1)
        )[0][0])
    )

    sim_min = df["embedding_similarity"].min()
    sim_max = df["embedding_similarity"].max()

    notas = []
    for idx, row in df.iterrows():
        desc = row.get("description", "").lower()
        _, job_techs = extract_skill_map(desc, tech_map)
        _, job_soft = extract_skill_map(desc, soft_map)

        nota = compute_final_score(
            row["embedding_similarity"],
            sim_min,
            sim_max,
            job_techs,
            job_soft,
            curr_techs,
            curr_soft
        )
        notas.append(nota)

    df["final_score"] = notas

    df_sorted = df.sort_values(by="final_score", ascending=False)

    for idx, row in df_sorted.iterrows():
        print("=" * 70)
        print(f"📌 Job #{idx}")
        print(f"🏷️  Title: {row.get('title', 'N/A')}")
        print(f"🏢 Company: {row.get('company_name', 'N/A')}")
        print(f"🔧 Tech Stacks Found: {row.get('tech_stacks_found', [])}")
        print(f"📊 Embedding Similarity: {row['embedding_similarity']:.4f}")
        print(f"🏆 Final Score: {row['final_score']:.2f}")
        print("-" * 70)
        print(f"📝 Description:\n{row.get('description','')[:500]}...")
        print("=" * 70)
        print()

    return df_sorted


Função que mostra stacks extraidas do curriculo

In [56]:
import json
from pathlib import Path
import re


def extract_skill_map(description: str, skill_map: dict):
    desc = description.lower()
    found = {}
    for macro, variations in skill_map.items():
        for v in variations:
            if v in desc:
                found.setdefault(macro, []).append(v)
    return desc, list(found.keys())


def load_support_maps():
    folder = Path(".")
    with open(folder / "tech_stack.json", "r", encoding="utf-8") as f:
        tech_map = json.load(f)
    with open(folder / "soft_skills.json", "r", encoding="utf-8") as f:
        soft_map = json.load(f)

    tech_map = {k.lower(): [v.lower() for v in vals] for k, vals in tech_map.items()}
    soft_map = {k.lower(): [v.lower() for v in vals] for k, vals in soft_map.items()}
    return tech_map, soft_map


def print_curriculo_info(curriculo: str):
    tech_map, soft_map = load_support_maps()

    curr_clean = re.sub(r"\s+", " ", curriculo).strip().lower()

    _, techs = extract_skill_map(curr_clean, tech_map)
    _, softs = extract_skill_map(curr_clean, soft_map)

    print("=" * 70)
    print("📄 CURRÍCULO ANALISADO\n")
    print(curriculo[:700], "...\n")

    print("🔧 TECH STACKS DETECTADAS:")
    for t in techs:
        print(f" - {t}")

    print("\n🤝 SOFT SKILLS DETECTADAS:")
    for s in softs:
        print(f" - {s}")

    print("=" * 70)

    return {"techs": techs, "softs": softs}

### Feedback Pontos da vaga

In [64]:
import os
import pandas as pd
from pathlib import Path
from typing import Dict, Any

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from pydantic import BaseModel, Field


class ResumeFeedback(BaseModel):
    pontos_fortes: list[str] = Field(description="Pontos fortes do candidato")
    pontos_a_melhorar: list[str] = Field(description="Pontos fracos")
    sugestoes: list[str] = Field(description="Sugestões práticas de melhoria")


def comparar_curriculo_vaga_langchain(
    curriculo: str,
    path_df: str,
    vaga_id: str,
    model_name: str = "gpt-4.1-mini"
) -> Dict[str, Any]:

    load_dotenv()
    key = os.getenv("openai")

    if not key:
        raise ValueError("ERRO: variavel 'openai' não encontrada no .env")

    path_df = Path(path_df)
    df = pd.read_csv(path_df) if path_df.suffix == ".csv" else pd.read_json(path_df)

    vaga_row = df[df["id"].astype(str) == str(vaga_id)]
    if vaga_row.empty:
        raise ValueError(f"ID da vaga '{vaga_id}' não encontrado.")

    vaga_texto = str(vaga_row.iloc[0].get("description", ""))

    parser = PydanticOutputParser(pydantic_object=ResumeFeedback)

    llm = ChatOpenAI(
        model=model_name,
        api_key=key,
        temperature=0
    )

    prompt = ChatPromptTemplate.from_template("""
Você é um especialista em análise de compatibilidade entre currículos e vagas.
Compare exclusivamente o texto literal do CURRÍCULO e da VAGA.

REGRAS:
- Não invente informações.
- Não adicione tecnologias ou experiências que não existam no currículo.
- Pontos devem ser objetivos e técnicos.
- Não faça suposições.

FORMATO OBRIGATÓRIO:
{format_instructions}

CURRÍCULO:
\"\"\"{curriculo}\"\"\"

VAGA:
\"\"\"{vaga}\"\"\"

Retorne APENAS o JSON final.
""")

    chain = (
        {
            "curriculo": RunnablePassthrough(),
            "vaga": lambda x: x["vaga"],
            "format_instructions": lambda _: parser.get_format_instructions()
        }
        | prompt
        | llm
        | RunnableLambda(lambda x: parser.parse(x.content))
    )

    result = chain.invoke({
        "curriculo": curriculo,
        "vaga": vaga_texto
    })

    return result.model_dump()

### Adaptar curriculo a vaga

In [77]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


def _strip_code_fences(text: str) -> str:
    if not text:
        return ""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if len(lines) >= 2 and (lines[0].startswith("```") and lines[-1].startswith("```")):
            cleaned = "\n".join(lines[1:-1]).strip()
    return cleaned


def build_prompt(curriculo: str, vaga: str) -> str:
    return f"""
Você é um especialista em RH.
Sua tarefa: ADAPTAR o currículo abaixo para a vaga fornecida.

O currículo final deve ser entregue em **TEXTO PURO**, sem LaTeX, sem Markdown, sem JSON.

Regras obrigatórias:
- Não invente informações novas apenas .
- Use termos e palavras chaves iguais aos pedidos na vaga
- Use apenas dados que realmente existem no currículo original.
- Estruture o currículo final em TEXTO seguindo exatamente este modelo:

NOME
(cidade – telefone – email – linkedin se existir)

OBJETIVO PROFISSIONAL
(Resumo curto de 1–2 linhas alinhado à vaga)

EDUCAÇÃO
(Formação principal)

SKILLS
(Lista de habilidades técnicas e soft skills extraídas do currículo e alinhadas à vaga. adicione caso seja uma stack muito parecida com o que ja tem no curriculo)

EXPERIÊNCIA PROFISSIONAL
(Nome da empresa — Cargo)
- Bullet points objetivos focados em impacto e tecnologias usadas
(Repita caso exista mais experiências)

PROJETOS / ATIVIDADES
(Bullets curtos sobre projetos, estágios, atividades relevantes)

Observações:
- Apenas reorganize e reescreva o texto para ficar mais compatível com a vaga.
- Saída deve ser somente TEXTO PURO.

CURRÍCULO ORIGINAL:
\"\"\"{curriculo}\"\"\"

DESCRIÇÃO DA VAGA:
\"\"\"{vaga}\"\"\"
"""


def gerar_curriculo_adaptado_por_id(curriculo: str, path_df: str, vaga_id: str) -> str:
    load_dotenv()
    key = os.getenv("openai")

    if not key:
        raise ValueError("Chave 'openai' não encontrada no .env.")

    path_df = Path(path_df)
    df = pd.read_csv(path_df) if path_df.suffix == ".csv" else pd.read_json(path_df)

    vaga_row = df[df["id"].astype(str) == str(vaga_id)]
    if vaga_row.empty:
        raise ValueError(f"ID {vaga_id} não encontrado no arquivo.")

    vaga_texto = str(vaga_row.iloc[0].get("description", ""))

    prompt = build_prompt(curriculo, vaga_texto)

    llm = ChatOpenAI(
        model="gpt-4.1",
        temperature=0,
        api_key=key
    )

    response = llm.invoke(prompt)
    raw_text = response.content
    clean_text = _strip_code_fences(raw_text)

    return clean_text

## -------------------------------------------------------------------------

## Explorando API e langchain

### Request Aprimorada Apenas com vagas Tech

In [ ]:
#SALVAR
import requests
import json
import os

# Categorias realmente tech
TECH_CATEGORIES = [
    "software-development",
    "ai-ml",
    "data",
    "devops",
    "qa"
]

def get_remotive_tech_jobs():
    url = "https://remotive.com/api/remote-jobs"
    response = requests.get(url)
    data = response.json()

    raw_jobs = data.get("jobs", [])

    tech_jobs = []

    for job in raw_jobs:
        category_slug = job.get("category", "").lower().replace(" ", "-")

        if category_slug in TECH_CATEGORIES:
            tech_jobs.append(job)
    return tech_jobs


def save_to_json(data, filename="tech_jobs.json"):
    # Caminho final: ../data/raw/tech_jobs.json
    save_path = os.path.join("..", "data", "raw", filename)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Arquivo salvo em: {save_path}")


tech_jobs = get_remotive_tech_jobs()
save_to_json(tech_jobs)

print(f"Salvo {len(tech_jobs)} tech jobs em tech_jobs.json")

Arquivo salvo em: ../data/raw/tech_jobs.json
Salvo 346 tech jobs em tech_jobs.json


Como registrar embeddings no banco de dados de vetores

In [ ]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os

load_dotenv()

def buscar_similares(curriculo_embedding, top_k=50):
    """
    Recebe o embedding do currículo e retorna:
        - id da vaga
        - score de similaridade
    ordenados do mais similar para o menos similar.
    """

    # Conectar ao Pinecone
    pc = Pinecone(api_key=os.getenv("pinecone_key"))
    index = pc.Index("hirematch-jobs")

    # Query no Pinecone
    results = index.query(
        vector=curriculo_embedding,
        top_k=top_k,
        include_metadata=False  # apenas scores e IDs
    )

    # Montar lista de resultados em (id, similaridade)
    similares = []
    for match in results["matches"]:
        similares.append(
            (match["id"], match["score"])
        )

    return similares

## Tests

In [30]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os
import pandas as pd
from langchain_pinecone import PineconeVectorStore

curriculo_emb = embeddings.embed_query(curriculo)



In [31]:
def buscar_ids_similaridade(curriculo_emb, top_k=20):
    # Carregar .env
    load_dotenv()
    pinecone_key = os.getenv("pinecone_key")

    # Conectar ao Pinecone
    pc = Pinecone(api_key=pinecone_key)
    index = pc.Index("hirematch-jobs")

    # Query com embedding pronto
    res = index.query(
        vector=curriculo_emb,
        top_k=top_k,
        include_metadata=False  # não precisamos de metadata
    )

    # Converter para JSON com apenas id + similarity
    saida = []
    for match in res["matches"]:
        saida.append({
            "id": match["id"],
            "similarity": match["score"]
        })

    return saida

In [ ]:
curriculo = "Aaaaaa"

curriculo_emb = embeddings.embed_query(curriculo)
buscar_ids_similaridade(curriculo_emb, top_k=100)

[{'id': '2074126', 'similarity': 0.443107605},
 {'id': '2071315', 'similarity': 0.422237396},
 {'id': '2069583', 'similarity': 0.418930054},
 {'id': '2074124', 'similarity': 0.41576767},
 {'id': '2072959', 'similarity': 0.415569305},
 {'id': '2075375', 'similarity': 0.411909103},
 {'id': '2070234', 'similarity': 0.410110503},
 {'id': '2071328', 'similarity': 0.407053},
 {'id': '2075371', 'similarity': 0.40385437},
 {'id': '2067411', 'similarity': 0.400814056},
 {'id': '2069972', 'similarity': 0.400753081},
 {'id': '2071336', 'similarity': 0.398086548},
 {'id': '2060877', 'similarity': 0.396476775},
 {'id': '2072964', 'similarity': 0.392108917},
 {'id': '2069970', 'similarity': 0.391033173},
 {'id': '2071812', 'similarity': 0.389183044},
 {'id': '2067617', 'similarity': 0.388988495},
 {'id': '2069215', 'similarity': 0.387687713},
 {'id': '2067422', 'similarity': 0.385402679},
 {'id': '2071516', 'similarity': 0.383087158},
 {'id': '1591692', 'similarity': 0.383060396},
 {'id': '2065602',

In [36]:
# Supabase uppp

from supabase import create_client, Client
import json
import os
from dotenv import load_dotenv

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

def insert_job(job: dict):
    data = supabase.table("jobs").insert(job).execute()
    return data

# Carregar seu JSON
with open("../data/processed/jobs_processed.json", "r") as f:
    jobs = json.load(f)

for job in jobs:
    insert_job(job)

In [24]:
import subprocess
import tempfile
from pathlib import Path
import shutil


def find_pdflatex() -> str:
    """
    Retorna o caminho completo do pdflatex ou lança erro se não encontrado.
    """

    # 1. Tenta descobrir via PATH
    path = shutil.which("pdflatex")
    if path:
        return path

    # 2. Locais comuns no Mac (MacTeX)
    mac_paths = [
        "/Library/TeX/texbin/pdflatex",
        "/usr/texbin/pdflatex",
        "/opt/homebrew/bin/pdflatex",
    ]

    for p in mac_paths:
        if Path(p).exists():
            return p

    # 3. Locais comuns no Linux
    linux_paths = [
        "/usr/bin/pdflatex",
        "/usr/local/bin/pdflatex",
        "/bin/pdflatex",
    ]

    for p in linux_paths:
        if Path(p).exists():
            return p

    raise FileNotFoundError(
        """
❌ 'pdflatex' não encontrado no seu sistema.

Instale com:

👉 macOS:
   brew install mactex-no-gui

👉 Linux (Ubuntu/Debian):
   sudo apt update
   sudo apt install texlive texlive-latex-extra

👉 Docker:
   apt install texlive-full

Depois tente novamente.
"""
    )


def latex_to_pdf(latex_str: str, output_path: str) -> str:
    """
    Converte LaTeX para PDF usando pdflatex detectado automaticamente.
    """

    pdflatex = find_pdflatex()

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        tex_file = tmpdir / "doc.tex"

        tex_file.write_text(latex_str, encoding="utf-8")

        cmd = [
            pdflatex,
            "-interaction=nonstopmode",
            "-halt-on-error",
            str(tex_file)
        ]

        result = subprocess.run(
            cmd,
            cwd=tmpdir,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode != 0:
            raise RuntimeError(
                "Erro na compilação LaTeX:\n\n"
                + result.stdout
                + "\n"
                + result.stderr
            )

        pdf_out = Path(output_path)
        pdf_out.write_bytes((tmpdir / "doc.pdf").read_bytes())

        return str(pdf_out)

## -------------------------------------------------------------------------

# Final > Apresentar

In [ ]:
pipeline_remotive_tech_jobs() # faz o request 
pipeline("tech_jobs.json") # Trata o request > retira tech stacks > limpa html 

✔ Arquivo salvo em: /Users/gabriel_goncalves/Desktop/UFABC/Projeto_PLN/HireMatch-AI/notebook/tech_jobs.json
✔ Salvo 14 tech jobs em tech_jobs.json
✔ Arquivo salvo em: /Users/gabriel_goncalves/Desktop/UFABC/Projeto_PLN/HireMatch-AI/notebook/jobs_processed.json


In [ ]:
generate_embeddings_df("jobs_processed.json") #gera os embeddings para cada vaga

✔ CSV salvo em: /Users/gabriel_goncalves/Desktop/UFABC/Projeto_PLN/HireMatch-AI/notebook/tech_jobs_com_embedding.csv


,id,url,title,company_name,company_logo,category,job_type,publication_date,candidate_required_location,salary,description,tech_stacks_found,soft_skills_found,company_logo_url,embedding
0,2079713,https://remotive.com/remote-jobs/qa/qa-enginee...,QA Engineer,PatientNow,https://remotive.com/job/2079713/logo,QA,full_time,2025-11-21T20:50:23,USA,,"the roleas our founding qa engineer, you’ll bu...","{'frontend_frameworks': ['next.js'], 'testing'...","{'communication': ['communication'], 'attentio...",NaN,"[-0.028542302548885345, 0.00807858631014824, -..."
1,2070452,https://remotive.com/remote-jobs/software-deve...,Quantitative Research Team Lead (Completed),Apexver,https://remotive.com/job/2070452/logo,Software Development,full_time,2025-11-21T20:00:41,Worldwide,$180k + performance bonus,"role overview as the quantitative team lead, y...","{'languages': ['c', 'python'], 'others_general...","{'communication': ['communication'], 'teamwork...",NaN,"[-0.04162572696805, 0.0008698273450136185, -0...."
2,2079702,https://remotive.com/remote-jobs/qa/sr-perform...,Sr. Performance Tester,Tietoevry,https://remotive.com/job/2079702/logo,QA,full_time,2025-11-21T18:51:44,India,,company descriptionwe are developers of digita...,"{'languages': ['java', 'python'], 'cloud': ['a...","{'communication': ['communication'], 'creativi...",NaN,"[-0.040924716740846634, -0.008528684265911579,..."
3,2080480,https://remotive.com/remote-jobs/software-deve...,Senior Software Engineer - Platform Development,Tanium,https://remotive.com/job/2080480/logo,Software Development,full_time,2025-11-21T16:50:18,Canada,"c$200,000 to c$220,000",the basics: as a tanium senior software engine...,"{'languages': ['c', 'go'], 'cloud': ['aws', 'a...","{'teamwork': ['collaboration'], 'time_manageme...",NaN,"[-0.03670961409807205, -0.03647104650735855, -..."
4,2080479,https://remotive.com/remote-jobs/software-deve...,Staff Application Security Engineer,PlayStation Global,https://remotive.com/job/2080479/logo,Software Development,full_time,2025-11-21T16:50:17,USA,"$198,200 - $297,400 usd",why playstation? playstation isn’t just the be...,"{'languages': ['c', 'java', 'javascript', 'pyt...","{'communication': ['communication'], 'teamwork...",NaN,"[-0.010759716853499413, -0.022283703088760376,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,2054061,https://remotive.com/remote-jobs/software-deve...,Full Stack Developer,Getwingapp,https://remotive.com/job/2054061/logo,Software Development,full_time,2025-08-31T00:51:25,India,inr 15 lpa - 25 lpa,about uswing is seeking elite talent to join m...,"{'languages': ['go', 'javascript', 'php', 'pyt...","{'leadership': ['leadership'], 'work_style': [...",NaN,"[-0.03674701973795891, -0.016622858121991158, ..."
343,2053123,https://remotive.com/remote-jobs/qa/software-d...,Software Development Engineer in Test,"Bayesian Health, Inc.",https://remotive.com/job/2053123/logo,QA,full_time,2025-08-27T22:51:54,USA,,software development engineer in testin briefw...,"{'testing': ['cypress', 'selenium'], 'producti...","{'teamwork': ['collaboration'], 'problem_solvi...",NaN,"[0.00016767217312008142, 0.022296486422419548,..."
344,2052625,https://remotive.com/remote-jobs/software-deve...,Internal Tooling Engineer Lead,Protex%20AI,https://remotive.com/job/2052625/logo,Software Development,full_time,2025-08-27T22:50:54,Hungary,,"about us:at protex ai, we are at the forefront...","{'languages': ['go', 'javascript', 'python'], ...","{'communication': ['communication'], 'accounta...",NaN,"[-0.02581094391644001, 0.0005910902982577682, ..."
345,2052631,https://remotive.com/remote-jobs/software-deve...,Senior Fullstack Engineer,Ciklum,https://remotive.com/job/2052631/logo,Software Development,full_time,2025-08-27T22:50:53,Poland,,ciklum is looking for a senior engineer to joi...,"{'languages': ['java'], 'backend_frameworks': ...","{'teamwork': ['collaboration'], 'adaptability'...",NaN,"[-0.03331394121050835, -0.047001298516988754, ..."


In [ ]:
match_curriculo_com_df(
    "tech_jobs_com_embedding.csv",
    curriculo
) #> similaridade
#Compara o curriculo com todas as vagas criando uma logica de nota dando peso de 70% para similarity 20% Tech_stack e 10% Soft_Skills

📌 Job #87
🏷️  Title: Systems Analyst
🏢 Company: Dev.Pro
🔧 Tech Stacks Found: {'cloud': ['aws', 'azure'], 'devops': ['docker', 'k8s'], 'databases_nosql': ['mongodb'], 'infra': ['apache', 'linux', 'nginx'], 'monitoring': ['datadog', 'grafana', 'prometheus'], 'productivity': ['jira'], 'others_general': ['cloud', 'documentation', 'fintech', 'hardware', 'infrastructure', 'insurance', 'it support', 'retail', 'technical support', 'troubleshooting']}
📊 Embedding Similarity: 0.4285
🏆 Final Score: 98.00
----------------------------------------------------------------------
📝 Description:
🟢 are you in brazil, argentina or colombia? join us as we actively recruit in these locations, offering a comfortable remote environment. submit your cv in english, and we'll get back to you!we invite a responsible and eager-to-learn network operations center engineer to join our 24/7 support team, helping ensure the stability and performance of client applications and . our client provides retailers with a cent

,id,url,title,company_name,company_logo,category,job_type,publication_date,candidate_required_location,salary,description,tech_stacks_found,soft_skills_found,company_logo_url,embedding,embedding_similarity,final_score
87,2074126,https://remotive.com/remote-jobs/software-deve...,Systems Analyst,Dev.Pro,https://remotive.com/job/2074126/logo,Software Development,full_time,2025-11-11T14:50:40,"Brazil, Colombia, Argentina",NaN,"🟢 are you in brazil, argentina or colombia? jo...","{'cloud': ['aws', 'azure'], 'devops': ['docker...","{'communication': ['communication'], 'teamwork...",NaN,"[-0.04396640136837959, 0.002905098022893071, -...",0.428520,98.00
88,2074124,https://remotive.com/remote-jobs/software-deve...,Python Engineer,TechBiz Global GmbH,https://remotive.com/job/2074124/logo,Software Development,full_time,2025-11-11T14:50:39,Czech Republic,NaN,"at techbiz global, we are providing recruitmen...","{'languages': ['bash', 'python'], 'backend_fra...","{'communication': ['communication'], 'accounta...",NaN,"[-0.0362430214881897, -0.007382248993963003, -...",0.415435,96.25
238,2069583,https://remotive.com/remote-jobs/software-deve...,Senior Java Developer,Penguin Formula,https://remotive.com/job/2069583/logo,Software Development,contract,2025-10-14T08:50:40,Europe,NaN,company descriptionwecookit is an internationa...,"{'languages': ['go', 'java', 'python'], 'front...","{'communication': ['communication'], 'teamwork...",NaN,"[-0.04270102083683014, -0.023362260311841965, ...",0.413453,95.68
75,2075375,https://remotive.com/remote-jobs/software-deve...,Front-End Engineer,Trustly,https://remotive.com/job/2075375/logo,Software Development,full_time,2025-11-13T14:50:57,Americas,NaN,about the team: the developer experience team ...,"{'languages': ['java', 'javascript'], 'fronten...","{'teamwork': ['collaboration'], 'attention_to_...",NaN,"[-0.03586525470018387, 0.01705746166408062, -0...",0.409474,94.54
159,2071328,https://remotive.com/remote-jobs/qa/senior-qua...,Senior Quality Assurance Automation Engineer,AlphaSights,https://remotive.com/job/2071328/logo,QA,full_time,2025-10-31T00:50:32,Brazil,NaN,the role: alphasights is seeking a highly expe...,"{'languages': ['java', 'javascript', 'python',...","{'leadership': ['leadership'], 'time_managemen...",NaN,"[-0.03234066814184189, 0.020477455109357834, -...",0.407295,93.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,2072097,https://remotive.com/remote-jobs/qa/quality-an...,Quality Analyst and Data Manager,Spectrum Human Services,https://remotive.com/job/2072097/logo,QA,full_time,2025-11-06T20:50:38,USA,NaN,quality analyst and data manager about the rol...,"{'others_general': ['cloud', 'data analysis', ...","{'communication': ['communication'], 'time_man...",NaN,"[-0.03946186229586601, 0.0022459090687334538, ...",0.230476,29.93
57,2076757,https://remotive.com/remote-jobs/software-deve...,Web Developer,TeamUpdraft,https://remotive.com/job/2076757/logo,Software Development,full_time,2025-11-15T18:50:11,India,competitive salary (depending on experience),here’s who we are: at team updraft our dna is ...,"{'languages': ['javascript', 'php'], 'frontend...",{'teamwork': ['collaboration']},NaN,"[-0.05123739317059517, -0.0172605961561203, -0...",0.252880,29.69
66,2076752,https://remotive.com/remote-jobs/software-deve...,Senior iOS Software Engineer,Reddit,https://remotive.com/job/2076752/logo,Software Development,full_time,2025-11-14T16:50:19,USA,"$190,800 - $267,100 usd",reddit is a community of communities. it’s bui...,"{'languages': ['go', 'swift'], 'others_general...","{'leadership': ['coaching'], 'time_management'...",NaN,"[-0.030059363692998886, -0.03409489989280701, ...",0.249635,28.76
11,2079701,https://remotive.com/remote-jobs/qa/qa-documen...,QA Documentation Specialist,Albert B Sabin Vaccine Institute Inc,https://remotive.com/job/2079701/logo,QA,full_time,2025-11-21T10:50:55,USA,"$72,000 - $85,000",apply job type full-time description departmen...,"{'others_g

In [57]:
print_curriculo_info(curriculo)

📄 CURRÍCULO ANALISADO

GABRIEL VICTOR LIMA GONC¸ ALVES
+55 (11)94924-4811 ⋄ Santo Andr´e, SP
gabrielvgonc@gmail.com ⋄ www.linkedin.com/in/gabriel-victor-71187b223
OBJECTIVE
Seeking career growth opportunities in the field of software development.
EDUCATION
Bachelor of Computer Science, Universidade Federal do ABC Expected Graduation: 2026
SKILLS
Main Technical Skills Python, SQL, ETL, Java, Git
Soft Skills Communication, Adaptability, Analytical Thinking, Proactivity
Github Portfolio https://github.com/Netreck
Languages English, Portuguese
PROFESSIONAL EXPERIENCE
Intern – Bank of America Jun 2025 – Present
VP Global Technology – Tech Rotation Program
• Developed test automation and tools to support QA teams in pay ...

🔧 TECH STACKS DETECTADAS:
 - languages
 - frontend_frameworks
 - backend_frameworks
 - databases_sql
 - ml_ai
 - bi_analytics
 - others_general

🤝 SOFT SKILLS DETECTADAS:
 - communication
 - leadership
 - problem_solving
 - adaptability
 - time_management
 - work_style
 

{'techs': ['languages',
  'frontend_frameworks',
  'backend_frameworks',
  'databases_sql',
  'ml_ai',
  'bi_analytics',
  'others_general'],
 'softs': ['communication',
  'leadership',
  'problem_solving',
  'adaptability',
  'time_management',
  'work_style',
  'management',
  'others_general']}

In [67]:
resultado = comparar_curriculo_vaga_langchain(
    curriculo,
    path_df="tech_jobs_com_embedding.csv",
    vaga_id="2079713"
)

resultado

{'pontos_fortes': ['Experiência com desenvolvimento de automação de testes e ferramentas para suporte a equipes de QA.',
  'Conhecimento em linguagens e tecnologias relevantes como Java, Python, Git e SQL.',
  'Experiência prática em automação interna e otimização de processos.',
  'Experiência com análise e monitoramento de KPIs operacionais e construção de soluções de Machine Learning.',
  'Experiência com testes automatizados e desenvolvimento de pipelines ETL.',
  'Habilidade em comunicação, adaptabilidade, pensamento analítico e proatividade.',
  'Experiência em gestão de projetos e equipes, incluindo delegação de tarefas e acompanhamento de progresso.',
  'Conhecimento em modelagem de dados, visualização e implantação de modelos de Machine Learning via APIs.'],
 'pontos_a_melhorar': ['Não possui experiência comprovada de 3-5 anos em QA engineering ou software testing conforme requerido.',
  'Não há menção explícita de experiência com testes manuais e regressão em ambientes de QA.

In [78]:
resultado = gerar_curriculo_adaptado_por_id(
    curriculo,
    path_df="tech_jobs_com_embedding.csv",
    vaga_id="2079713"
)

print(resultado)

GABRIEL VICTOR LIMA GONÇALVES  
Santo André, SP – +55 (11)94924-4811 – gabrielvgonc@gmail.com – www.linkedin.com/in/gabriel-victor-71187b223

OBJETIVO PROFISSIONAL  
Atuar como QA Engineer, contribuindo para a definição e execução de processos de qualidade, automação de testes e garantia de releases confiáveis em ambientes de tecnologia e IA.

EDUCAÇÃO  
Bacharelado em Ciência da Computação, Universidade Federal do ABC – Conclusão prevista: 2026

SKILLS  
Python, Java, SQL, ETL, Git, Node.js, HTML, CSS, Test Automation, Processos de QA, Testes Manuais e Automatizados, Machine Learning, Data Visualization, Power BI, Excel, Comunicação, Adaptabilidade, Pensamento Analítico, Proatividade, Inglês, Português

EXPERIÊNCIA PROFISSIONAL  
Bank of America — Estagiário em Tecnologia (Tech Rotation Program)  
- Desenvolveu automação de testes e ferramentas para apoiar equipes de QA em sistemas de pagamentos.  
- Atuou em projetos internos de automação e otimização de processos.  
- Utilizou tecno